<a href="https://colab.research.google.com/github/tungduong03/Deep-Learning/blob/main/W2V_%2B_CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
import pandas as pd
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt

#for text pre-processing
import re, string
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import SnowballStemmer
from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer

nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
nltk.download('wordnet')

#for model-building
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import SGDClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, f1_score, accuracy_score, confusion_matrix
from sklearn.metrics import roc_curve, auc, roc_auc_score

# bag of words
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer

#for word embedding
import gensim
from gensim.models import Word2Vec #Word2Vec is mostly used for huge datasets

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [22]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/Code_Injection_Dataset

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/Code_Injection_Dataset


In [3]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import numpy as np
import json


class Vectorizer:
    def __init__(self, method='BOW', ngram_range=(1, 1), max_features=300, emb_fname='', word_index_fname=''):
        self.method = method
        if self.method == 'BOW':
            self.vectorizer = CountVectorizer(analyzer='word', input='content', ngram_range=ngram_range, max_features=max_features)
        elif self.method == 'TFIDF':
            self.vectorizer = TfidfVectorizer(analyzer='word', input='content', max_features=max_features)
        elif self.method == 'Word2Vec':
            self.max_features = max_features
            self.emb_fname = emb_fname
            self.word_index_fname = word_index_fname
        else:
            raise ValueError('Feature extraction method does not exist.')

    def feature_extraction(self, X_train, X_test):
        train_data = self.vectorizer.fit_transform(X_train).toarray()
        test_data = self.vectorizer.transform(X_test).toarray()
        return train_data, test_data

    def get_word_index(self):
        word2id = json.load(open(self.word_index_fname, 'r'))
        return word2id

    def get_embedding_matrix(self):
        np.random.seed(0)
        word2id = self.get_word_index()
        embedding_matrix = np.random.uniform(-0.25, 0.25, [len(word2id) + 1, self.max_features])
        with open(self.emb_fname, 'r', encoding='utf-8') as f:
            for line in f:
                content = line.split(' ')
                if content[0] in word2id:
                    embedding_matrix[word2id[content[0]]] = np.array(list(map(float, content[1:])))
        return embedding_matrix

In [4]:
df_train = pd.read_csv('dataset_capec.csv')

**Word2Vec Feature Extraction**

In [5]:
import pandas as pd
import numpy as np
import string
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from gensim.models import Word2Vec
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Embedding, Conv1D, MaxPooling1D, Flatten, Dense
from tensorflow.keras.models import Sequential
from sklearn.model_selection import train_test_split

In [10]:
X_train, X_test, y_train, y_test = train_test_split(df_train["text"],
                                                  df_train["label"],
                                                  test_size=0.2,
                                                  shuffle=True)

In [11]:
from gensim.models import Word2Vec

sentences = [sentence.split() for sentence in X_train]
w2v_model = Word2Vec(sentences, window=5, min_count=5, workers=4)

In [8]:
import numpy as np

def vectorize(sentence):
    words = sentence.split()
    words_vecs = [w2v_model.wv[word] for word in words if word in w2v_model.wv]
    if len(words_vecs) == 0:
        return np.zeros(100)
    words_vecs = np.array(words_vecs)
    return words_vecs.mean(axis=0)

X_train = np.array([vectorize(sentence) for sentence in X_train])
X_test = np.array([vectorize(sentence) for sentence in X_test])

In [12]:
from gensim.models import Word2Vec

# Tokenize sentences
sentences = [sentence.split() for sentence in X_train]  # X_train là dữ liệu văn bản

# Train Word2Vec model
w2v_model = Word2Vec(sentences, vector_size=100, window=5, min_count=5, workers=4)
embedding_dim = w2v_model.vector_size  # Kích thước vector nhúng (100)


In [13]:
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Tokenize sentences
tokenizer = Tokenizer()
tokenizer.fit_on_texts(X_train)  # X_train là danh sách câu văn bản
word_index = tokenizer.word_index  # Từ điển {từ: chỉ số}

# Tạo embedding matrix
embedding_matrix = np.zeros((len(word_index) + 1, embedding_dim))
for word, i in word_index.items():
    if word in w2v_model.wv:
        embedding_matrix[i] = w2v_model.wv[word]


In [14]:
# Convert sentences to sequences of indices
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

# Pad sequences to the same length
max_length = 100  # Độ dài tối đa cho mỗi câu
X_train_padded = pad_sequences(X_train_seq, maxlen=max_length, padding='post')
X_test_padded = pad_sequences(X_test_seq, maxlen=max_length, padding='post')


In [15]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, MaxPooling1D, Flatten, Dense, Dropout

model = Sequential([
    # Embedding layer: Sử dụng Word2Vec embeddings
    Embedding(input_dim=len(word_index) + 1,
              output_dim=embedding_dim,
              weights=[embedding_matrix],
              input_length=max_length,
              trainable=False),  # Đóng băng (không huấn luyện lại Word2Vec embeddings)

    # Convolutional layer
    Conv1D(filters=128, kernel_size=5, activation='relu'),

    # MaxPooling layer
    MaxPooling1D(pool_size=2),

    # Flatten layer
    Flatten(),

    # Fully connected (Dense) layers
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(9, activation='softmax')  # Lớp đầu ra cho 8 loại
])

# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])


/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [16]:
# Huấn luyện mô hình
from sklearn.preprocessing import LabelEncoder

# Create a LabelEncoder object
label_encoder = LabelEncoder()

# Fit the encoder to your training labels
label_encoder.fit(y_train)

# Transform your training and testing labels
y_train_encoded = label_encoder.transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

print(np.unique(y_train_encoded))


[0 1 2 3 4 5 6 7 8]


In [17]:
# Huấn luyện mô hình

from tensorflow.keras.utils import to_categorical

# Convert labels to one-hot encoding
# Changed line to use the encoded labels instead of original y_train
y_train_encoded = to_categorical(y_train_encoded, num_classes=9)
y_test_encoded = to_categorical(y_test_encoded, num_classes=9)


history = model.fit(X_train_padded, y_train_encoded,
                    epochs=10,
                    batch_size=32,
                    validation_split=0.2)

# Đánh giá mô hình
test_loss, test_acc = model.evaluate(X_test_padded, y_test_encoded)
print(f"Test accuracy: {test_acc}")

Epoch 1/10
11836/11836 ━━━━━━━━━━━━━━━━━━━━ 41s 3ms/step - accuracy: 0.4165 - loss: 1.3624 - val_accuracy: 0.4250 - val_loss: 1.3336
Epoch 2/10
11836/11836 ━━━━━━━━━━━━━━━━━━━━ 34s 3ms/step - accuracy: 0.4211 - loss: 1.3348 - val_accuracy: 0.4249 - val_loss: 1.3332
Epoch 3/10
11836/11836 ━━━━━━━━━━━━━━━━━━━━ 46s 4ms/step - accuracy: 0.4239 - loss: 1.3346 - val_accuracy: 0.4249 - val_loss: 1.3330
Epoch 4/10
11836/11836 ━━━━━━━━━━━━━━━━━━━━ 70s 3ms/step - accuracy: 0.4226 - loss: 1.3341 - val_accuracy: 0.4247 - val_loss: 1.3334
Epoch 5/10
11836/11836 ━━━━━━━━━━━━━━━━━━━━ 34s 3ms/step - accuracy: 0.4243 - loss: 1.3339 - val_accuracy: 0.3872 - val_loss: 1.3346
Epoch 6/10
11836/11836 ━━━━━━━━━━━━━━━━━━━━ 33s 3ms/step - accuracy: 0.4236 - loss: 1.3334 - val_accuracy: 0.4249 - val_loss: 1.3332
Epoch 7/10
11836/11836 ━━━━━━━━━━━━━━━━━━━━ 40s 3ms/step - accuracy: 0.4249 - loss: 1.3319 - val_accuracy: 0.4247 - val_loss: 1.3334
Epoch 8/10
11836/11836 ━━━━━━━━━━━━━━━━━━━━ 43s 3ms/step - accuracy: 

In [20]:
from sklearn.metrics import classification_report
import numpy as np

# Dự đoán nhãn từ mô hình
y_pred = model.predict(X_test_padded)
y_pred_labels = np.argmax(y_pred, axis=1)  # Chuyển one-hot thành nhãn
y_test_labels = np.argmax(y_test_encoded, axis=1)  # Chuyển one-hot thành nhãn thật
label_names = label_encoder.classes_

# Tạo báo cáo đánh giá
report = classification_report(y_test_labels, y_pred_labels, target_names=label_names)
print(report)


3699/3699 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step
                                        precision    recall  f1-score   support

                          000 - Normal       0.63      0.02      0.03     45225
                  126 - Path Traversal       0.00      0.00      0.00      3502
         153 - Input Data Manipulation       0.00      0.00      0.00       303
         194 - Fake the Source of Data       0.45      0.00      0.01     11151
                  242 - Code Injection       1.00      0.00      0.00      2809
           272 - Protocol Manipulation       0.00      0.00      0.00      1384
310 - Scanning for Vulnerable Software       0.71      0.39      0.50       475
          34 - HTTP Response Splitting       0.00      0.00      0.00      3852
                    66 - SQL Injection       0.42      1.00      0.59     49659

                              accuracy                           0.43    118360
                             macro avg       0.36      0.16      0.13    11

/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
